# linalg-solve-batched — ex1: solve a batch of 2x2 systems

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `linalg-solve-batched`. Running the final beacon cell reports progress against the `PyTorch: Batched linalg.solve` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Batched linalg.solve` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`linalg-solve-batched`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "linalg-solve-batched"
DD_SUBTOPIC = "PyTorch: Batched linalg.solve"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `t.linalg.solve` — batched form

`t.linalg.solve(A, b)` solves `A x = b` for `x`. It accepts **arbitrary leading batch dimensions** as long as the last two axes of `A` are square:

- `A: (..., n, n)` and `b: (..., n)` → `x: (..., n)`
- `A: (..., n, n)` and `b: (..., n, k)` → `x: (..., n, k)` (k right-hand sides)

Internally it factors each `(n, n)` slice once and substitutes — vastly faster than a Python loop over `t.linalg.inv` + matmul, and numerically better-behaved.

**Failure mode.** If any leading slice's `A` is singular (or even very close to it), the call raises `LinAlgError`. The standard workaround is the *singular-matrix-mask trick*: detect singular slices via `t.linalg.det(A).abs() < eps`, overwrite them with the identity so the solve succeeds, then mask their results out of the final answer.

### Exercise 1 — solve a batch of 2x2 systems

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `t.linalg.solve` with leading batch dimensions to solve `K` independent 2x2 systems in a single call, returning a `(K, 2)` stack of solutions.
> Keywords: linalg, solve, batched, shape
> ```

**KCs targeted:** `linalg-solve-leading-batch`, `linalg-solve-shape-contract`

Implement `ex1_batched_solve_2x2(A, b)`.

- `A` has shape `(K, 2, 2)` — `K` square coefficient matrices.
- `b` has shape `(K, 2)` — `K` right-hand-side vectors.
- Return shape `(K, 2)`: the `k`th row is the solution `x_k` to `A[k] @ x_k == b[k]`.

**Hint.** This is *exactly* what `t.linalg.solve` does — pass `A` and `b` as-is. No loops, no per-slice indexing. The output shape mirrors `b`.

Assume all matrices are non-singular (the singular-matrix-mask trick is a separate drill).

After solving, the test verifies by recomputing `A @ x` and comparing against `b` to floating-point tolerance.

In [ ]:
def ex1_batched_solve_2x2(A: Tensor, b: Tensor) -> Tensor:
    return t.linalg.solve(A, b)


<details><summary>Solution</summary>

```python
def ex1_batched_solve_2x2(A: Tensor, b: Tensor) -> Tensor:
    return t.linalg.solve(A, b)
```

**The shape contract.** `t.linalg.solve(A, b)` with `A: (..., n, n)` and `b: (..., n)` returns `x: (..., n)`. The leading dims of `A` and `b` must agree (or be broadcastable); the last two of `A` must be square and equal to `b`'s last dim.

**Why not loop.** A Python loop over `K` would re-dispatch to BLAS once per system, paying per-call overhead. The batched call fuses the LU factorizations into a single C-level loop — for `K=64` already 10x+ faster, and the gap widens with larger `K`.

**Multiple right-hand sides.** If `b` has shape `(..., n, m)` instead of `(..., n)`, you get one solution column per `m`-slice — same factorization, different substitutions. Useful for problems like "compute the inverse" (set `b = I`).

**Singular slices crash the whole call.** That's the failure mode the `singular-matrix-mask-trick` drill addresses.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()